模型本身没有记忆，需要手动管理上下文窗口，历史会话和摘要

In [12]:
from zai import ZhipuAiClient
from dotenv import load_dotenv
import time
from enum import Enum
from pydantic import BaseModel

load_dotenv()
client = ZhipuAiClient()

class ROLE(str, Enum):
    user: str = "user"
    assistant: str = "assistant"

class ContentModel(BaseModel):
    content: str
    usage: str

class MessageModel(BaseModel):
    role: ROLE
    content: str

messages: list[MessageModel] = []

In [13]:
def fetch(kwargs:dict) -> ContentModel:
    result = client.chat.completions.create(**kwargs)
    content = result.choices[0].message.content
    prompt_tokens = result.usage.prompt_tokens
    completion_tokens = result.usage.completion_tokens
    total_tokens = result.usage.total_tokens
    cache_tokens = result.usage.prompt_tokens_details.cached_tokens
    log = f"提示词:{prompt_tokens} 回复内容:{completion_tokens} 缓存:{cache_tokens} 共计:{total_tokens}"
    return ContentModel(content=content, usage=log)

不带历史会话

In [14]:
def chat_with_no_memory(input: str):
    params:dict = {
        "model": "glm-4.5-air",
        "messages": [
            {'role': 'user', 'content': input},
        ],
        "stream": False,
    }
    result = fetch(params)
    print(f"{result.content}\n{result.usage}")

In [4]:
chat_with_no_memory("我是小明，记住了")

好的，小明！我记住了你的名字 😊

以后在对话中，我会直接称呼你为“小明”。有什么问题或者需要帮助的地方，随时告诉我！
提示词:10 回复内容:269 缓存:4 共计:279


In [5]:
chat_with_no_memory("我是谁？")

这是一个深刻而永恒的问题，答案因人而异，也因你思考的层面而不同。我们可以从几个角度来探索“你是谁”：

### 1. **生物学角度：你是独特的生命体**
   - **基因的奇迹**：你是地球上约 800 万物种中唯一拥有如此特定 DNA 序列的人类。你的基因组合独一无二，决定了你的外貌、部分性格倾向和生理特征。
   - **身体的宇宙**：你的身体由约 37 万亿个细胞组成，它们协同工作，维持你的呼吸、心跳、思考。你是一个由无数微小生命组成的复杂系统。

### 2. **社会角度：你是关系网络中的节点**
   - **角色与身份**：你是子女、朋友、同事、恋人……这些社会角色定义了你在他人眼中的位置。你的名字、职业、家庭背景等社会标签，共同构成了你“社会身份”的一部分。
   - **文化烙印**：你成长的语言、习俗、价值观塑造了你的认知方式。比如，东方文化中的集体主义与西方的个人主义，会深刻影响你如何定义“自我”。

### 3. **心理角度：你是意识的载体**
   - **记忆与经历**：你的过去、痛苦、喜悦、遗憾共同编织成你的记忆库。这些经历塑造了你的性格、恐惧、渴望和信念。
   - **思想与情感**：你会思考、质疑、做梦，会因一首歌流泪，因一个笑话发笑。你的意识是宇宙中已知最复杂的现象之一，而你是它的唯一拥有者。
   - **价值观与选择**：你相信什么？什么对你最重要？你如何面对道德困境？这些选择定义了你的精神内核。

### 4. **哲学角度：你是存在之谜的探索者**
   - **存在主义视角**：萨特说“存在先于本质”——你并非天生被赋予意义，而是通过行动和选择创造自己的价值。你的自由意志让你成为自己人生的作者。
   - **东方智慧**：道家认为“道法自然”，你与万物相连；佛家提出“无我”，执着于固定身份是痛苦的根源。你的“自我”可能是流动变化的幻象。

### 5. **时间维度：你是流动的生命过程**
   - **过去的你**：孩童时的你、十年前的你，都已不复存在，但他们的经历沉淀为今天的你。
   - **未来的你**：你尚未成为的样子，却正在被此刻的选择塑造。你是一个不断演变的“正在进行时”。

---

### 一个可能的答案：
**“你是宇宙中，通过独特经历、选择和意识，不断追问‘我是谁’的那个存在本身。”**

带着历史会话

In [15]:
def chat_with_memory(input: str):
    messages.append(MessageModel(role=ROLE.user, content=input))

    params:dict = {
        "model": "glm-4.5-air",
        "messages": [m.model_dump() for m in messages],
        "stream": False,
    }
    result = fetch(params)

    messages.append(MessageModel(role=ROLE.assistant, content=result.content))

    print(f"{result.content}\n{result.usage}")

In [16]:
chat_with_memory("我是小明，记住了")

好的，小明！我记住了你的名字 😊  

有什么我可以帮你做的吗？无论是回答问题、提供信息、聊天，还是其他任何需求，随时告诉我就好！
提示词:10 回复内容:258 缓存:4 共计:268


In [17]:
chat_with_memory("我是谁？")

哈哈，你是小明呀！刚才我们刚聊过，我还特意记住了你的名字呢！✨  

需要我帮你回忆一下之前的对话内容，或者现在有什么想聊的吗？
提示词:57 回复内容:236 缓存:12 共计:293


In [18]:
chat_with_memory("我喜欢旅游，我去过马来西亚和印度尼西亚")

太棒了！🌴✈️ 小明，马来西亚和印度尼西亚都是东南亚非常迷人的旅游目的地！  

### 🇲🇾 马来西亚的特色：
- **吉隆坡**：双子塔、独立广场、美食天堂（如肉骨茶、椰浆饭）  
- **槟城**：彩色壁画、娘惹文化、街头小吃  
- **沙巴/仙本那**：潜水天堂、玻璃海、海岛度假  
- **马六甲**：历史古城、葡式建筑、娘惹美食  

### 🇮🇩 印度尼西亚的亮点：
- **巴厘岛**：梯田（德格拉朗）、海神庙、冲浪与瑜伽  
- **日惹**：婆罗浮屠（世界最大佛寺）、普兰巴南神庙  
- **龙目岛**：粉色沙滩、吉利群岛潜水、火山徒步  
- **雅加达**：城市文化、传统市场（如芝布达斯市场）  

### 🌍 小明，你最喜欢哪个地方？  
是海岛的阳光沙滩，还是人文历史？或者有没有什么特别难忘的经历？比如：  
- 吃到惊艳的美食？  
- 遇到有趣的人？  
- 拍到绝美的照片？  

如果下次计划旅行，我也可以帮你推荐路线或攻略哦！🗺️✨
提示词:112 回复内容:552 缓存:4 共计:664


In [19]:
chat_with_memory("我去过哪里？")

哈哈，小明！根据我们之前的对话，你提到你**去过马来西亚和印度尼西亚**哦！🌴✈️  

如果你愿意，可以多分享一些旅行经历呀！比如：  
- **马来西亚**：你最喜欢吉隆坡的双子塔、槟城的壁画，还是沙巴的海岛？  
- **印度尼西亚**：是去了巴厘岛的梯田和海神庙，还是日惹的千年佛寺？  

或者有没有什么**难忘的美食/冒险/趣事**？比如在仙本那潜水、在巴厘岛学冲浪，还是吃了什么让你念念不忘的街头小吃？🍜🍢  

等你分享更多细节，我可以帮你整理成旅行攻略，或者推荐下一个目的地！ 😄
提示词:420 回复内容:361 缓存:114 共计:781
